# Metadata Filters and Index Management

Production corpora are not homogeneous: API docs, security policies, per-tenant data, and deprecated versions all live in the same retrieval system. **Metadata** attaches structured fields to each chunk so you can **filter** before or during vector search — "only search `security-docs`" or "only `tenant_acme`".

**Index management** covers the full lifecycle: idempotent upserts, deletes when sources change, collection versioning, and **full re-index** when embedding models or chunk strategies change.

This notebook covers metadata schema design, Chroma `where` filters, CRUD operations, blue/green collection migrations, and re-indexing checklists with live demonstrations.

Prerequisites: **06. Chroma — Collections, Ingest, and Query**.

In [ ]:
import chromadb
import numpy as np

np.set_printoptions(precision=4, suppress=True)

---

## 1. Why Metadata Matters

### 1.1 Without Filters

A query about **JWT expiry** might retrieve an Alembic paragraph if it mentions "tokens" in passing — wrong domain, high embedding similarity noise.

### 1.2 With Filters

Restrict search to `source=security-docs` or `doc_type=policy` so neighbors are drawn from the right **subset**.

| Metadata field | Example use |
|----------------|-------------|
| `source` | File path, URL, repo name |
| `doc_type` | `api`, `policy`, `faq`, `runbook` |
| `tenant_id` | Multi-tenant SaaS isolation |
| `version` | `2025-03` — deprecate old chunks |
| `language` | `en`, `de` |
| `section` | Markdown header title |
| `visibility` | `internal` vs `public` |

### 1.3 Design at Ingest Time

Metadata should be planned when you define chunk records (notebook 04). Retrofitting filters after launch requires re-ingest and client changes.

---

## 2. Metadata Schema Guidelines

| Guideline | Reason |
|-----------|--------|
| **Flat keys** | Chroma metadata is JSON-like key–value |
| **Stable enums** | `doc_type=api` not free-text paragraphs |
| **No huge payloads** | Store ids, not full HTML in metadata |
| **Consistent naming** | `snake_case` across pipeline |
| **Indexable fields only** | Fields you will actually filter on |

Chroma metadata values are typically **strings, numbers, or booleans** — check current docs for supported types.

---

## 3. Chroma `where` Filter Syntax

Pass `where` to **`query`** and **`get`**:

### 3.1 Shorthand Equality

```python
where={"source": "security-docs"}
# equivalent to {"source": {"$eq": "security-docs"}}
```

### 3.2 Common Operators

| Operator | Meaning | Example |
|----------|---------|--------|
| `$eq` | Equal | `{"tenant_id": {"$eq": "acme"}}` |
| `$ne` | Not equal | |
| `$in` | In list | `{"doc_type": {"$in": ["policy", "faq"]}}` |
| `$nin` | Not in list | |
| `$and` | All conditions | `{"$and": [{...}, {...}]}` |
| `$or` | Any condition | `{"$or": [{...}, {...}]}` |

Syntax evolves — verify against [Chroma documentation](https://docs.trychroma.com/) for your installed version.

---

## 4. Pre-Filter vs Post-Filter

| Approach | Behavior |
|----------|----------|
| **Pre-filter (DB)** | ANN search only within matching metadata — efficient |
| **Post-filter (app)** | Retrieve top-k globally, then drop wrong metadata — can miss hits |

Prefer **database-level `where`** when available. Post-filtering is a fallback when filters are complex or unsupported.

---

## 5. Index Management Overview

```
  CREATE collection (versioned name)
       ↓
  INGEST chunks (batch add / upsert)
       ↓
  QUERY with optional where
       ↓
  UPDATE — upsert by id, delete stale ids
       ↓
  MIGRATE — new collection + re-embed + cutover
       ↓
  DELETE old collection
```

| Operation | API | Notes |
|-----------|-----|-------|
| Upsert | `add` same `id` | Overwrites vector + document + metadata |
| Delete by id | `delete(ids=[...])` | Remove stale chunks |
| Delete by filter | `delete(where=...)` | Bulk removal |
| Get by filter | `get(where=...)` | Audit / debug |
| Update metadata | `update` (if available) or upsert | Fix labels without re-embed |

---

## 6. Re-Indexing and Blue/Green Collections

Trigger **full re-index** when:

- Embedding **model** changes (`3-small` → `3-large`)
- **`dimensions`** changes
- **Chunk strategy** changes materially
- Metadata schema requires fields that cannot be defaulted

**Blue/green pattern:**

1. Build `docs_v2` collection alongside `docs_v1`
2. Run eval (notebook 09) on `v2`
3. Point app config to `v2`
4. Keep `v1` briefly for rollback
5. Delete `v1` after confidence period

---

## 7. Demonstration Setup

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"

DOCS = [
    {"id": "a1", "text": "JWT access tokens expire after 15 minutes.", "source": "security-docs", "doc_type": "policy", "tenant_id": "acme"},
    {"id": "a2", "text": "POST /auth/login returns a bearer token.", "source": "api-docs", "doc_type": "api", "tenant_id": "acme"},
    {"id": "a3", "text": "Refresh tokens rotate on each use.", "source": "security-docs", "doc_type": "policy", "tenant_id": "acme"},
    {"id": "a4", "text": "Alembic revision files live in versions/.", "source": "db-docs", "doc_type": "api", "tenant_id": "acme"},
    {"id": "b1", "text": "Tenant beta uses OAuth2 client credentials flow.", "source": "security-docs", "doc_type": "policy", "tenant_id": "beta"},
    {"id": "b2", "text": "FAQ: reset password via email link.", "source": "faq", "doc_type": "faq", "tenant_id": "acme"},
]


def embed(texts: list[str]) -> list[list[float]]:
    r = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    ordered = sorted(r.data, key=lambda x: x.index)
    return [d.embedding for d in ordered]


def fresh_col(name: str):
    client = chromadb.Client()
    try:
        client.delete_collection(name)
    except Exception:
        pass
    return client.create_collection(name, metadata={"hnsw:space": "cosine"})


col = fresh_col("filtered_demo")
texts = [d["text"] for d in DOCS]
col.add(
    ids=[d["id"] for d in DOCS],
    documents=texts,
    embeddings=embed(texts),
    metadatas=[{
        "source": d["source"],
        "doc_type": d["doc_type"],
        "tenant_id": d["tenant_id"],
    } for d in DOCS],
)
print("Ingested:", col.count())

---

## 8. Demonstration: Unfiltered vs Equality Filter

In [ ]:
query = "How do authentication tokens work?"
q_emb = embed([query])

print("=== Unfiltered top 3 ===")
r_all = col.query(query_embeddings=q_emb, n_results=3, include=["documents", "metadatas", "distances"])
for doc, meta, dist in zip(r_all["documents"][0], r_all["metadatas"][0], r_all["distances"][0]):
    print(f"  d={dist:.4f} [{meta['source']:14s}] {doc}")

print("\n=== where source=security-docs ===")
r_sec = col.query(
    query_embeddings=q_emb,
    n_results=3,
    where={"source": "security-docs"},
    include=["documents", "metadatas", "distances"],
)
for doc, meta, dist in zip(r_sec["documents"][0], r_sec["metadatas"][0], r_sec["distances"][0]):
    print(f"  d={dist:.4f} [{meta['source']}] {doc}")

---

## 9. Demonstration: `$in` and `$and` Filters

In [ ]:
print("=== doc_type in [policy, faq] ===")
r_in = col.query(
    query_embeddings=q_emb,
    n_results=5,
    where={"doc_type": {"$in": ["policy", "faq"]}},
    include=["metadatas", "documents"],
)
for doc, meta in zip(r_in["documents"][0], r_in["metadatas"][0]):
    print(f"  [{meta['doc_type']:6s}] {doc[:60]}...")

print("\n=== tenant=acme AND source=security-docs ===")
r_and = col.query(
    query_embeddings=q_emb,
    n_results=5,
    where={"$and": [{"tenant_id": "acme"}, {"source": "security-docs"}]},
    include=["metadatas", "documents"],
)
for doc, meta in zip(r_and["documents"][0], r_and["metadatas"][0]):
    print(f"  tenant={meta['tenant_id']}  {doc}")

---

## 10. Demonstration: `get` with Metadata Filter

In [ ]:
rows = col.get(where={"tenant_id": "beta"}, include=["documents", "metadatas"])
print(f"Beta tenant chunks: {len(rows['ids'])}")
for i, doc_id in enumerate(rows["ids"]):
    print(f"  {doc_id}: {rows['documents'][i]}")

---

## 11. Demonstration: Delete and Upsert Lifecycle

In [ ]:
print("Before delete:", col.count())
col.delete(ids=["a4"])
print("After delete a4:", col.count())

new_text = "Alembic autogenerate compares models to the live database schema."
col.add(
    ids=["a4"],
    documents=[new_text],
    embeddings=embed([new_text]),
    metadatas=[{"source": "db-docs", "doc_type": "api", "tenant_id": "acme"}],
)
print("After upsert a4:", col.count())
print("a4 text:", col.get(ids=["a4"])["documents"][0])

---

## 12. Demonstration: Delete by Filter

In [ ]:
mgmt_col = fresh_col("delete_filter_demo")
mgmt_col.add(
    ids=[d["id"] for d in DOCS],
    documents=[d["text"] for d in DOCS],
    embeddings=embed([d["text"] for d in DOCS]),
    metadatas=[{"source": d["source"], "doc_type": d["doc_type"], "tenant_id": d["tenant_id"]} for d in DOCS],
)
print("Before:", mgmt_col.count())

mgmt_col.delete(where={"tenant_id": "beta"})
print("After delete tenant=beta:", mgmt_col.count())
remaining = mgmt_col.get(include=["metadatas"])
print("Tenants left:", sorted({m["tenant_id"] for m in remaining["metadatas"]}))

---

## 13. Demonstration: Collection Versioning (Blue/Green)

In [ ]:
client = chromadb.Client()
for name in ("docs_v1", "docs_v2"):
    try:
        client.delete_collection(name)
    except Exception:
        pass

v1 = client.create_collection("docs_v1")
v1.add(
    ids=["x1"],
    documents=["Old embedding model chunk about migrations."],
    embeddings=embed(["Old embedding model chunk about migrations."]),
    metadatas=[{"index_version": "v1"}],
)

v2 = client.create_collection("docs_v2")
v2.add(
    ids=["x1"],
    documents=["Re-embedded chunk: Alembic manages SQLAlchemy schema migrations."],
    embeddings=embed(["Re-embedded chunk: Alembic manages SQLAlchemy schema migrations."]),
    metadatas=[{"index_version": "v2"}],
)

ACTIVE = "docs_v2"  # config flag in production
active_col = client.get_collection(ACTIVE)
q = embed(["database migrations"])
hit = active_col.query(query_embeddings=q, n_results=1, include=["documents", "metadatas"])
print(f"Active collection: {ACTIVE}")
print("Hit:", hit["documents"][0][0])
print("Meta:", hit["metadatas"][0][0])

---

## 14. Stable Chunk IDs for Idempotent Ingest

```python
import hashlib

def chunk_id(source: str, chunk_index: int) -> str:
    return f"{source}#{chunk_index}"

def content_hash(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()[:12]
```

Re-running ingest with the same ids **updates** changed chunks without duplicates. Use content hashes to detect whether re-embed is needed.

In [ ]:
import hashlib


def chunk_id(source: str, index: int) -> str:
    return f"{source}#{index}"


source = "db-docs.md"
chunks = [
    "Alembic manages migrations.",
    "Revision files in versions/.",
]
ids = [chunk_id(source, i) for i in range(len(chunks))]
hashes = [hashlib.sha256(c.encode()).hexdigest()[:12] for c in chunks]

for cid, h, c in zip(ids, hashes, chunks):
    print(f"{cid}  hash={h}  {c}")

---

## 15. Multi-Tenant Pattern

| Pattern | When |
|---------|------|
| **Single collection + `tenant_id` filter** | Many small tenants |
| **Collection per tenant** | Strict isolation, large tenants |
| **Separate DB instances** | Enterprise compliance |

Always include `tenant_id` in the **query filter** for shared collections — never rely on the LLM to self-restrict.

---

## 16. Re-Indexing Checklist

1. Freeze config: model, dimensions, chunk size, overlap
2. Create `collection_v{N+1}`
3. Re-chunk all sources from canonical store (not old chunks)
4. Batch embed and upsert with stable ids
5. Run Recall@k eval (notebook 09) vs previous version
6. Update `ACTIVE_COLLECTION` env / feature flag
7. Monitor query latency and retrieval quality
8. Delete old collection after rollback window

---

## 17. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| No tenant filter | Data leak across customers | `where` on every query |
| Random ids each ingest | Duplicates | Stable chunk_id scheme |
| Filter on non-ingested field | Empty results | Schema at ingest |
| In-place model swap | Broken geometry | New collection |
| Delete without backup | Lost index | Blue/green + rollback |
| Post-filter only | Missed relevant docs in subset | DB-level `where` |

---

## 18. Summary

**Metadata filters** scope vector search to the right documents — by source, type, tenant, or version. Use Chroma **`where`** on `query`, `get`, and `delete` with equality, **`$in`**, and **`$and`** operators.

**Index management**: upsert by stable **ids**, delete stale rows, version collections for blue/green migrations, and **full re-index** when embedding models change.

Next: **08. Vector Database Landscape and Trade-offs**.